In [55]:
import pandas as pd

In [56]:
df = pd.DataFrame(
    [
        ["Alice","国語", 87],
        ["Alice","数学", 72],
        ["Bob","国語", 65],
        ["Bob","数学", 92],
    ],
    columns=["Name", "Subject", "Point"],
)

df[df.Subject == "数学"]

,Name,Subject,Point
1,Alice,数学,72
3,Bob,数学,92


###### 実務寄り練習問題

In [57]:
#1 ライブラリー読み込み
import pandas as pd
from pathlib import Path

Path("output").mkdir(exist_ok=True)

In [58]:
#2 品質データ作成
df_quality = pd.DataFrame(
    [
         ["2026-06", "Lamp-A", 12000, 18, 10000, 25, 8000, 4],
        ["2026-06", "Lamp-B", 8500, 4, 7000, 5, 6000, 18],
        ["2026-07", "Lamp-A", 11000, 30, 9500, 10, 9000, 8],
        ["2026-07", "Lamp-C", 9000, 8, 6000, 18, 5200, 3],
    ],
    columns=[
        "month",
        "product",
        "melmb_production",
        "melmb_defects",
        "wse_production",
        "wse_defects",
        "gse_production",
        "gse_defects",
    ],
)

df_quality

,month,product,melmb_production,melmb_defects,wse_production,wse_defects,gse_production,gse_defects
0,2026-06,Lamp-A,12000,18,10000,25,8000,4
1,2026-06,Lamp-B,8500,4,7000,5,6000,18
2,2026-07,Lamp-A,11000,30,9500,10,9000,8
3,2026-07,Lamp-C,9000,8,6000,18,5200,3


In [59]:
df_info = df_quality.iloc[:, 0:2]
df_info

,month,product
0,2026-06,Lamp-A
1,2026-06,Lamp-B
2,2026-07,Lamp-A
3,2026-07,Lamp-C


In [60]:
df_ppm = df_info.copy()
df_ppm

,month,product
0,2026-06,Lamp-A
1,2026-06,Lamp-B
2,2026-07,Lamp-A
3,2026-07,Lamp-C


In [61]:
#5 ppm calc
df_ppm["melmb_ppm"] = (
    df_quality["melmb_defects"] / df_quality["melmb_production"] * 1000000
).round(1)

df_ppm["wse_ppm"] = (
    df_quality["wse_defects"] / df_quality["wse_production"] * 1000000
).round(1)

df_ppm["gse_ppm"] = (
    df_quality["gse_defects"] / df_quality["gse_production"] * 1000000
).round(1)

df_ppm

,month,product,melmb_ppm,wse_ppm,gse_ppm
0,2026-06,Lamp-A,1500.0,2500.0,500.0
1,2026-06,Lamp-B,470.6,714.3,3000.0
2,2026-07,Lamp-A,2727.3,1052.6,888.9
3,2026-07,Lamp-C,888.9,3000.0,576.9


###### 6.最大ppm 最悪拠点 calc

In [62]:
df_ppm_values = df_ppm.iloc[:, 2:]

df_ppm["max_ppm"] = df_ppm_values.max(axis=1)

df_ppm["worst_site"] = df_ppm_values.idxmax(axis=1)

df_ppm["worst_site"] = df_ppm["worst_site"].str.replace("_ppm", "", regex=False)

df_ppm

,month,product,melmb_ppm,wse_ppm,gse_ppm,max_ppm,worst_site
0,2026-06,Lamp-A,1500.0,2500.0,500.0,2500.0,wse
1,2026-06,Lamp-B,470.6,714.3,3000.0,3000.0,gse
2,2026-07,Lamp-A,2727.3,1052.6,888.9,2727.3,melmb
3,2026-07,Lamp-C,888.9,3000.0,576.9,3000.0,wse


In [63]:
#7 危険ランク作成
df_ppm["danger_rank"] = np.where(
    df_ppm["max_ppm"] >= 3000,
    "危険！！",
    np.where(
        df_ppm["max_ppm"] >= 1000,
        "注意",
        "通常"
    )
)
df_ppm

,month,product,melmb_ppm,wse_ppm,gse_ppm,max_ppm,worst_site,danger_rank
0,2026-06,Lamp-A,1500.0,2500.0,500.0,2500.0,wse,注意
1,2026-06,Lamp-B,470.6,714.3,3000.0,3000.0,gse,危険！！
2,2026-07,Lamp-A,2727.3,1052.6,888.9,2727.3,melmb,注意
3,2026-07,Lamp-C,888.9,3000.0,576.9,3000.0,wse,危険！！


In [64]:
# 8 条件で行を絞り込むから作成